# Transformations and Actions

Spark has two kinds of DataFrame operations:

| | Transformations | Actions |
|---|---|---|
| What they do | Describe a new DataFrame built from an existing one | Ask Spark for a result |
| When they run | **Lazily**: nothing runs yet, Spark only adds a step to the plan | **Right away**: they start a Spark job |
| Return | A new DataFrame | A value, or output on screen or disk |
| Examples | `select`, `where`/`filter`, `groupBy`, `orderBy`, `limit` | `show`, `count`, `collect`, `take`, `write`, `display` |

DataFrames are **immutable**. A transformation never changes the original DataFrame. It always returns a new one.

## 1. Top 3 zip codes with the most calls

**Question:** which 3 zip codes account for the most fire calls?

**Approach:** group the calls by `Zipcode`, count the rows in each group, sort from highest to lowest, and keep the first 3.

`%sql` on the first line makes this cell run as SQL instead of Python. In a SQL cell, comments start with `--`, not `#`.

> Outside a Databricks notebook, this `%sql` cell returns a DataFrame, so the output shows only its schema (`DataFrame[...]`), not the rows.

In [9]:
%sql
-- Columns to return: call type, zip code, and the number of calls in each group
select callType, Zipcode, count(*) as total_calls
-- Source table (catalog.schema.table)
from dev.spark_db.sf_fire_calls
-- Ignore rows where the zip code is missing
where Zipcode is not null
-- One group per (call type, zip code) pair
group by CallType, Zipcode
-- Biggest counts first
order by total_calls desc
-- Keep only the top 3
limit 3

,callType,Zipcode,total_calls
0,Medical Incident,94102,16130
1,Medical Incident,94103,14775
2,Medical Incident,94110,9995


## 2. The same query with the DataFrame API

The SQL query above can be broken into 3 parts:

1. **Read** the source table into a DataFrame
2. **Transform** it: select, filter, group and count, sort, limit (these are all lazy)
3. **Act** on it: show the result (this is what actually runs the job)

Each SQL clause maps to a DataFrame method:

| SQL | DataFrame API |
|---|---|
| `from table` | `spark.read.table("...")` |
| `select cols` | `.select("col1", "col2")` |
| `where cond` | `.where("cond")` or `.filter(...)` |
| `group by ... count(*)` | `.groupBy(...).count()` |
| `order by ... desc` | `.orderBy("col", ascending=False)` |
| `limit n` | `.limit(n)` |

### Step 1: read the data

In [1]:
# Get a SparkSession. Here it connects to Databricks through Databricks Connect
# (see the startup banner in the output).
from pyspark.shell import spark

# Load the table as a DataFrame. Only the schema is read here, not the data.
fire_df = spark.read.table("dev.spark_db.sf_fire_calls")

# Outside a Databricks notebook, display() prints only the schema, like
# DataFrame[...] below. Use fire_df.show(5) to see actual rows.
display(fire_df)

Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /__ / .__/\_,_/_/ /_/\_\   version 4.1.0
      /_/

Using Python version 3.12.16 (main, Nov 11 2026 00:15:59)
Connected to the Databricks Connect server 'dbc-18a9dd91-0021.cloud.databricks.com'.
SparkSession available as 'spark'.


,CallNumber,UnitID,IncidentNumber,CallType,CallDate,WatchDate,CallFinalDisposition,AvailableDtTm,Address,City,Zipcode,Battalion,StationArea,Box,OriginalPriority,Priority,FinalPriority,ALSUnit,CallTypeGroup,NumAlarms,UnitType,UnitSequenceInCallDispatch,FirePreventionDistrict,SupervisorDistrict,Neighborhood,Location,RowID,Delay
0,20110016,T13,2003235,Structure Fire,2002-01-11,2002-01-10,Other,2002-01-11 01:51:44,2000 Block of CALIFORNIA ST,SF,94109,B04,38,3362,3,3,3,False,None,1,TRUCK,2,4,5,Pacific Heights,"(37.7895840679362, -122.428071912459)",020110016-T13,2.950000
1,20110022,M17,2003241,Medical Incident,2002-01-11,2002-01-10,Other,2002-01-11 03:01:18,0 Block of SILVERVIEW DR,SF,94124,B10,42,6495,3,3,3,True,None,1,MEDIC,1,10,10,Bayview Hunters Point,"(37.7337623673897, -122.396113802632)",020110022-M17,4.700000
2,20110023,M41,2003242,Medical Incident,2002-01-11,2002-01-10,Other,2002-01-11 02:39:50,MARKET ST/MCALLISTER ST,SF,94102,B03,01,1455,3,3,3,True,None,1,MEDIC,2,3,6,Tenderloin,"(37.7811772186856, -122.411699931232)",020110023-M41,2.433333
3,20110032,E11,2003250,Vehicle Fire,2002-01-11,2002-01-10,Other,2002-01-11 04:16:46,APPLETON AV/MISSION ST,SF,94110,B06,32,5626,3,3,3,False,None,1,ENGINE,1,6,9,Bernal Heights,"(37.7388432849018, -122.423948785199)",020110032-E11,1.500000
4,20110043,B04,2003259,Alarms,2002-01-11,2002-01-10,Other,2002-01-11 06:01:58,1400 Block of SUTTER ST,SF,94109,B04,03,3223,3,3,3,False,None,1,CHIEF,2,4,2,Western Addition,"(37.7872890372638, -122.424236212664)",020110043-B04,3.483333
5,20110072,T08,2003279,Structure Fire,2002-01-11,2002-01-11,Other,2002-01-11 08:03:26,BEALE ST/FOLSOM ST,SF,94105,B03,35,2122,3,3,3,False,None,1,TRUCK,2,3,6,Financial District/South Beach,"(37.7886866619654, -122.392722833778)",020110072-T08,1.750000
6,20110125,E33,2003301,Alarms,2002-01-11,2002-01-11,Other,2002-01-11 09:46:44,0 Block of FARALLONES ST,SF,94112,B09,33,8324,3,3,3,False,None,1,ENGINE,2,9,11,Oceanview/Merced/Ingleside,"(37.7140353531157, -122.454117149916)",020110125-E33,2.716667
7,20110130,E36,2003304,Alarms,2002-01-11,2002-01-11,Other,2002-01-11 09:58:53,600 Block of POLK ST,SF,94102,B02,03,3114,3,3,3,False,None,1,ENGINE,1,2,6,Tenderloin,"(37.7826266328595, -122.41915582123)",020110130-E36,1.783333
8,20110197,E05,2003343,Medical Incident,2002-01-11,2002-01-11,Other,2002-01-11 12:06:57,1500 Block of WEBSTER ST,SF,94115,B04,05,3513,3,3,3,False,None,1,ENGINE,1,4,5,Japantown,"(37.784958590666, -122.431435274503)",020110197-E05,1.516667
9,20110215,E06,2003348,Medical Incident,2002-01-11,2002-01-11,Other,2002-01-11 13:08:40,DIAMOND ST/MARKET ST,SF,94114,B05,06,5415,3,3,3,False,None,1,ENGINE,1,5,8,Castro/Upper Market,"(37.7618954753708, -122.437298717721)",020110215-E06,2.766667


### Step 2: transformations

Each line creates a new DataFrame from the one before. **No data is processed yet.** Spark only builds up a plan of steps, so this cell finishes immediately.

In [2]:
# SELECT callType, Zipcode: keep only the 2 columns we need
df_1 = fire_df.select("callType", "Zipcode")

# WHERE: drop rows with a missing value
# Note: the SQL version filters on Zipcode, but this filters on callType
df_2 = df_1.where("callType is not null")

# GROUP BY + COUNT(*): one row per (callType, Zipcode) pair, with a "count" column
df_3 = df_2.groupBy("callType", "Zipcode").count()

# ORDER BY count DESC: biggest counts first
df_4 = df_3.orderBy("count", ascending=False)

# LIMIT 3: keep the top 3 rows
df_5 = df_4.limit(3)

### Step 3: action

`.show()` is an **action**. Calling it makes Spark optimize the whole plan from Step 2 and run it as a job, then print the result. This is the first time any data is actually read.

In [3]:
# Action: runs the job and prints the top 3 rows as a text table
df_5.show()

+----------------+-------+-----+                                                                                                         
|        callType|Zipcode|count|
+----------------+-------+-----+
|Medical Incident|  94102|16130|
|Medical Incident|  94103|14775|
|Medical Incident|  94110| 9995|
+----------------+-------+-----+



## 3. Seeing the plan with `explain`

`explain()` prints the plan Spark built for a DataFrame. It does **not** run a job.
`mode="extended"` (same as `explain(True)`) prints every stage of the plan:

| Section | What it shows |
|---|---|
| Parsed Logical Plan | Your code as written, not checked yet |
| Analyzed Logical Plan | Checked against the catalog: columns now have types and IDs (`callType#10211`) |
| Optimized Logical Plan | The plan after Spark's optimizations |
| Physical Plan | How it will actually run on the machines |

See notebook `04-catalyst-optimizer` for the full explanation of these stages.

In [8]:
# Print every stage of the plan for df_5 (no job is run)
df_5.explain(mode="extended")

== Parsed Logical Plan ==
'GlobalLimit 3
+- 'LocalLimit 3
   +- 'Sort ['count DESC NULLS LAST], true
      +- 'Aggregate ['callType, 'Zipcode], ['callType, 'Zipcode, 'count(1) AS count#10119]
         +- 'Filter isnotnull('callType)
            +- 'Project ['callType, 'Zipcode]
               +- 'UnresolvedRelation [dev, spark_db, sf_fire_calls], [], false

== Analyzed Logical Plan ==
callType: string, Zipcode: string, count: bigint
GlobalLimit 3
+- LocalLimit 3
   +- Sort [count#10119L DESC NULLS LAST], true
      +- Aggregate [callType#10211, Zipcode#10218], [callType#10211, Zipcode#10218, count(1) AS count#10119L]
         +- Filter isnotnull(callType#10211)
            +- Project [callType#10211, Zipcode#10218]
               +- SubqueryAlias dev.spark_db.sf_fire_calls
                  +- Relation dev.spark_db.sf_fire_calls[CallNumber#10208,UnitID#10209,IncidentNumber#10210,CallType#10211,CallDate#10212,WatchDate#10213,CallFinalDisposition#10214,AvailableDtTm#10215,Address#10216,C

### What the output tells us

- **Parsed plan:** names start with `'` (like `'callType`) and the table is an `UnresolvedRelation`: Spark hasn't checked anything yet.
- **Analyzed plan:** the table is resolved to `dev.spark_db.sf_fire_calls` and each column gets a type and an ID.
- **Optimized plan:** `Filter` moved **below** `Project`, so the filter is applied as early as possible (predicate pushdown).
- **Physical plan:**
  - `PhotonScan ... ReadSchema: struct<CallType:string,Zipcode:string>`: only the **2 needed columns** are read from the files, not all 28 (column pruning).
  - `DataFilters: [isnotnull(CallType)]`: the filter is applied **while reading**.
  - `PhotonTopK(limit=3 ...)`: sort + limit are combined into one "top 3" step, so Spark never sorts the full result.
  - `AdaptiveSparkPlan isFinalPlan=false`: Adaptive Query Execution can still adjust the plan while it runs.
  - `Photon...`: Photon is Databricks' fast native engine. On plain open-source Spark you would see names like `HashAggregate` and `FileScan` instead.

## 4. Chaining transformations (composable approach)

Instead of saving each step in its own variable (`df_1` → `df_2` → ... → `df_5`), you can **chain** the transformations in a single statement.
It does the same thing as the code above, but it's shorter and easier to read top to bottom.

The parentheses `( ... )` around the chain let you split it over several lines without `\` at the end of each line.
Spark builds exactly the same plan either way, so the two styles run equally fast.

In [10]:
# Same query as df_1 ... df_5, written as one chain.
# Still only transformations: nothing runs yet.
Result_df = (
    fire_df.select("callType", "Zipcode")              # SELECT callType, Zipcode
        .where("callType is not null")                 # WHERE callType IS NOT NULL
        .groupBy("callType", "Zipcode").count()        # GROUP BY + COUNT(*)
        .orderBy("count", ascending=False)             # ORDER BY count DESC
        .limit(3)                                      # LIMIT 3
)

In [11]:
# Outside Databricks, display() prints only the schema (see output below).
# Use Result_df.show() to run the job and see the rows.
display(Result_df)

,callType,Zipcode,count
0,Medical Incident,94102,16130
1,Medical Incident,94103,14775
2,Medical Incident,94110,9995


## Key takeaways

- **Transformations** (`select`, `where`, `groupBy`, `orderBy`, `limit`) are lazy: they only build a plan, so they return instantly.
- **Actions** (`show`, `count`, `collect`, `write`) run the plan as a Spark job.
- Spark optimizes the whole chain at once: here it read only 2 columns, filtered while reading, and turned sort + limit into a single "top 3" step.
- SQL and the DataFrame API produce the same plan, so neither one is faster.
- `explain()` shows the plan without running anything.
- Separate variables and a single chain give the same result; chaining is shorter.